In [1]:
from pathlib import Path

from modules import SequenceRepresentation as sr
from modules import training

2025-03-06 22:55:16.294988: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-06 22:55:17.075540: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-06 22:55:17.638652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741301718.202215  181101 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741301718.366946  181101 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-06 22:55:19.737539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [3]:
wd = Path("/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/")
experiment_dirs = [f for f in wd.iterdir() if f.is_dir()]
print(experiment_dirs)

[PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsSydhK562Gata2UcdUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/wgEncodeAw

In [4]:
glob_seqs = {}
for ed in experiment_dirs:
    if not (ed / 'evaluator_test.json').exists():
        continue

    assert (ed / 'test_sequences_0.json').exists()
    testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
    seqdict = {s.id: 0 for g in testdata for s in g} # count how many times each sequence was hit
    evaluator = training.loadMultiTrainingEvaluation(
        str(ed / 'evaluator_test.json'),
        testdata
    )

    assert len(evaluator.trainings) == 1
    tr = evaluator.trainings[0]
    for link in tr.links:
        for occs in link.occs: # list of list of occurrences
            for occ in occs:
                assert occ.sequence.id in seqdict
                seqdict[occ.sequence.id] += 1

    for s in seqdict:
        if s not in glob_seqs:
            glob_seqs[s] = 0
        glob_seqs[s] += seqdict[s]

In [5]:
nseqs = len(glob_seqs.keys())
nseqs_hit = len([k for k in glob_seqs.keys() if glob_seqs[k] > 0])
nmatches = sum(glob_seqs.values())

print(f"Number of sequences: {nseqs}")
print(f"Number of sequences with hits: {nseqs_hit} | ratio: {nseqs_hit/nseqs:.2f}")
print(f"Number of matches: {nmatches} | ratio: {nmatches/nseqs:.2f}")

Number of sequences: 11997
Number of sequences with hits: 9970 | ratio: 0.83
Number of matches: 47782 | ratio: 3.98
